# 03 - Exploracion y Perfilamiento de Datos
## Analisis Descriptivo de la Capa Bronce

---

### Objetivo de este notebook

Antes de transformar los datos hacia capas superiores, es fundamental conocer
en detalle **que tenemos, cuanto tenemos y en que estado se encuentra**.
Este notebook realiza un **perfilamiento exhaustivo** (data profiling) de las
tablas de la capa Bronce.

### Que se analiza?

| Aspecto | Descripcion |
|---------|-------------|
| Estructura | Esquema de columnas, tipos de datos, dimensiones |
| Completitud | Porcentaje de valores nulos o vacios por columna |
| Distribucion | Como se distribuyen los valores clave (resultados, tipos de ensayo) |
| Volumetria | Cantidad de registros por medicamento |
| Propiedades | Valores fisicoquimicos de los compuestos |

### Por que es importante?

El perfilamiento de datos permite:
- Identificar **problemas de calidad** antes de que se propaguen a capas superiores
- Tomar decisiones informadas sobre **filtros y transformaciones** necesarias
- Documentar las **caracteristicas del dataset** para futura referencia
- Establecer **lineas base** (baselines) contra las cuales comparar ingestas futuras

---
## 1. Carga de Tablas Bronce

Cargamos las dos tablas de la capa Bronce y mostramos su volumetria basica.

In [ ]:
from pyspark.sql import functions as F  # Funciones de transformacion de Spark SQL

In [ ]:
# Cargar las tablas de la capa Bronce como DataFrames de Spark
df_props = spark.table("pubchem_bronce.propiedades_compuestos")  # Propiedades fisicoquimicas
df_bio = spark.table("pubchem_bronce.bioactividad")              # Resultados de bioensayos

# Contar registros para dimensionar el dataset
total_compuestos = df_props.count()
total_bioactividad = df_bio.count()

print(f"Compuestos en Bronce:     {total_compuestos}")
print(f"Bioactividades en Bronce: {total_bioactividad:,}")

---
## 2. Estructura de las Tablas

Revisamos el **esquema** (schema) de cada tabla para entender:
- Nombres de las columnas disponibles
- Tipos de datos inferidos por Spark (string, long, double, etc.)
- Si las columnas permiten valores nulos (nullable)

Esto es critico para planificar las conversiones de tipos en la capa Plata.

In [ ]:
# Mostrar el esquema completo de la tabla de propiedades
# Las columnas mantienen los nombres originales de PubChem (CID, MolecularFormula, etc.)
print("--- Esquema: propiedades_compuestos ---")
df_props.printSchema()

# Mostrar el esquema de la tabla de bioactividad
# Las columnas ya fueron renombradas a snake_case en el notebook 02
print("--- Esquema: bioactividad ---")
df_bio.printSchema()

---
## 3. Analisis de Valores Nulos en Bioactividad

Este analisis es crucial para determinar la **completitud** de cada campo.
Los campos con alta proporcion de nulos pueden requerir:
- Imputacion de valores predeterminados
- Exclusion del modelo analitico
- Documentacion como limitacion del dataset

Se clasifican los campos segun su nivel de completitud:
- **COMPLETO**: 0% nulos
- **BUENO**: menos del 10% nulos
- **PARCIAL**: entre 10% y 50% nulos
- **CRITICO**: mas del 50% nulos

In [ ]:
# Obtener el total de registros para calcular porcentajes
total = df_bio.count()

# Encabezado del reporte de nulidad
print("Campo                        Nulos/Vacios    Porcentaje")
print("-" * 60)

# Iterar sobre cada columna de la tabla de bioactividad
for nombre_columna in df_bio.columns:
    # Omitir columnas de trazabilidad (empiezan con _)
    if nombre_columna.startswith("_"):
        continue

    # Contar registros donde la columna es nula o es una cadena vacia
    nulos = df_bio.filter(
        (F.col(nombre_columna).isNull()) | (F.col(nombre_columna) == "")
    ).count()

    # Calcular el porcentaje de nulidad
    porcentaje = round(nulos * 100 / total, 1)

    # Asignar clasificacion de completitud basada en umbrales
    if porcentaje == 0:
        indicador = "COMPLETO"
    elif porcentaje < 10:
        indicador = "BUENO"
    elif porcentaje < 50:
        indicador = "PARCIAL"
    else:
        indicador = "CRITICO"

    # Mostrar resultado formateado con alineacion
    print(f"  {nombre_columna:25s} {nulos:>10,}    {porcentaje:>5.1f}%  [{indicador}]")

---
## 4. Distribucion de Resultados de Bioensayos

El campo `activity_outcome` indica el resultado del bioensayo:
- **Active**: El compuesto mostro actividad biologica
- **Inactive**: El compuesto no mostro actividad
- **Inconclusive**: El resultado no fue determinante
- **Probe**: Compuesto utilizado como sonda molecular

La proporcion entre activos e inactivos es un indicador fundamental
de la selectividad farmacologica de los compuestos.

In [ ]:
# Agrupar por resultado de actividad y calcular la cantidad y porcentaje de cada tipo
print("--- Resultados de bioensayos (activity_outcome) ---")
df_bio.groupBy("activity_outcome").agg(
    F.count("*").alias("cantidad"),                                   # Conteo de registros
    F.round(F.count("*") * 100 / total, 1).alias("porcentaje")       # Porcentaje del total
).orderBy(F.desc("cantidad")).show()  # Ordenar de mayor a menor cantidad

---
## 5. Actividades por Compuesto

Analizamos cuantos ensayos biologicos tiene cada medicamento y cuantos
resultaron activos vs inactivos. Esto permite identificar:
- Medicamentos con **mayor cobertura** de ensayos (mas estudiados)
- Medicamentos con **mayor tasa de actividad** (mas selectivos)
- Posibles **desbalances** en la cantidad de datos por compuesto

In [ ]:
# Agrupar bioactividad por medicamento y calcular metricas clave
print("--- Bioensayos por medicamento ---")
df_bio.groupBy("nombre_comun", "cid").agg(
    F.count("*").alias("total_ensayos"),  # Total de ensayos para el compuesto
    # Contar ensayos con resultado 'Active'
    F.sum(F.when(F.col("activity_outcome") == "Active", 1).otherwise(0)).alias("activos"),
    # Contar ensayos con resultado 'Inactive'
    F.sum(F.when(F.col("activity_outcome") == "Inactive", 1).otherwise(0)).alias("inactivos")
).orderBy(F.desc("total_ensayos")).show(truncate=False)  # Ordenar por total de ensayos

---
## 6. Tipos de Ensayo

El campo `assay_type` clasifica los ensayos por su metodologia:
- **Screening**: Ensayos de alto rendimiento (High Throughput Screening - HTS)
- **Confirmatory**: Ensayos de confirmacion de resultados iniciales
- **Summary**: Resumenes de multiples ensayos
- **Panel**: Ensayos tipo panel con multiples objetivos

La distribucion por tipo de ensayo indica la composicion metodologica del dataset.

In [ ]:
# Agrupar por tipo de ensayo y contar registros
print("--- Distribucion por tipo de ensayo (assay_type) ---")
df_bio.groupBy("assay_type").count().orderBy(F.desc("count")).show()

---
## 7. Propiedades Fisicoquimicas de los Compuestos

Revisamos las propiedades principales de los 15 medicamentos.
Estas propiedades son importantes para:
- Evaluar la **biodisponibilidad oral** (Regla de Lipinski)
- Comparar **complejidad molecular** entre compuestos
- Entender la **solubilidad** (XLogP) y **polaridad** del compuesto

In [ ]:
# Mostrar las propiedades fisicoquimicas mas relevantes de cada medicamento
# Estas columnas conservan los nombres originales de PubChem en la capa Bronce
print("--- Propiedades fisicoquimicas de los medicamentos ---")
df_props.select(
    "nombre_comun",        # Nombre del medicamento en espanol
    "CID",                 # Identificador unico de PubChem
    "MolecularFormula",    # Formula molecular (ej: C9H8O4 para Aspirina)
    "MolecularWeight",     # Peso molecular en Daltons
    "XLogP",               # Coeficiente de particion (lipofilicidad)
    "Complexity"           # Complejidad molecular (indice numerico)
).show(truncate=False)

print("Exploracion de datos completada.")